# Demo Notebook to use SSLimPy 

## Initial Setup

In [ ]:
import os
import sys

sys.path.append("../")

envkey = "OMP_NUM_THREADS"
# Set this environment variable to the number of available cores in your machine,
# to get a fast execution of the Einstein Boltzmann Solver
print("The value of {:s} is: ".format(envkey), os.environ.get(envkey))
os.environ[envkey] = str(12)
print("The value of {:s} is: ".format(envkey), os.environ.get(envkey))

In [ ]:
import astropy.units as u
import matplotlib.pyplot as plt
from copy import copy, deepcopy
import numpy as np
import seaborn
from getdist.gaussian_mixtures import GaussianND
from getdist import plots
from scipy.optimize import curve_fit

In [ ]:
# seaborn.set_theme(rc={'axes.edgecolor': 'black', 'xtick.color': 'black', 'ytick.color': 'black',})
import niceplots.utils as nicepl
nicepl.initPlot()

In [ ]:
Cs = seaborn.color_palette("colorblind")
Cp = seaborn.color_palette("Paired")
Cs

## Choose model parameters and save them in dictionaries

In [ ]:
# Settings will be the global settings that should not change during a given run.
# For the full list see the Configuration class in interface

settings = {
    "code":"class", # The Einstein--Boltzman solver that should be used
    "do_RSD" : False, # If RSD should be considerd
    "nonlinearRSD" : True, # If you want to add FOG to the RSD
    "QNLpowerspectrum": False, # Use dewiggled power spectrum (vlasov approximation of nonlinear structure formation)
    "FoG_damp" : "ISTF_like", # The particular parametrization for the FOG. Check PowerSpectrum for the full list
    "halo_model_PS" : True, # If the cosmological shotnoise should be computed from the halo model 
    "output" : ["Power spectrum", "Covariance"], # What output one wants (here power spectrum and Gaussian covariance only)
    "kmin": 1e-4 * u.Mpc**-1,
    "kmax": 50 * u.Mpc**-1,
    "nk": 200,
}

In [ ]:
# Cosmological parameters for the fiducial cosmology can be set like this
# Depending on your test you can fix them vary them
cosmodict={
    "h": 0.677,
    "Omegam": 0.309167,
    "Omegab": 0.04903,
    "sigma8":0.8222,
    "ns":0.96824,
    "mnu":0.06,
    "Neff":3.044,
}

In [ ]:
1/myastro.Mpch

In [ ]:
# Parameters that enter your halo model. Typically they are not changed but you could
halodict={
    "halo_tracer" : "clustering", # Computes all halo quantities from the matter field - neutrinos
    "hmf_model": "ST", # Sheth--Tormann halo mass function
    "concentration": "Diemer19", # Diemer19 halo concentration relation
    "bias_model": "ST99",
}

In [ ]:
# Parameters for Astrophysics.
astrodict_CO={
    "model_type": "ML",
    "model_name": "TonyLi",
    "model_par": {
        "alpha": 1.11,
        "beta": 0.6,
        "dMF": 1 * u.Msun * u.yr**-1 * u.Lsun**-1,
        "sig_SFR":0,
        "SFR_file": "sfr_release.dat",
        "do_quench": False,
    },
    "sigma_scatter" : 0.37
} # CO21

astrodict_CII={
    "model_type": "ML",
    "model_name": "SilvaCII",
    "model_par": {
        "a": 0.8475,
        "b": 7.2203,
        "SFR_file": "sfr_release.dat",
        "do_quench": False,
    },
    "sigma_scatter" : 0.37
}

In [ ]:
# Parameters for the Survey specifications
def get_specs(nu):
        z = 1
        nuObs = nu / (z + 1)
        DeltaDeltanu = 0.1

        surveyspecs = {
                "Tsys_NEFD": 0 * u.uK, #System temperature for instrumental shotnoise
                "Nfeeds": 19,
                "tobs": 1300 * u.h,
                "nD": 1, # Observational parameters
                "beam_FWHM": 0. * u.arcmin,
                "nu":  nu,
                "nuObs": nuObs, # Observed Frequency
                "Delta_nu": DeltaDeltanu * nuObs, # Frequency Bin
                "dnu": 0. * u.MHz, # Spectrograph resolution
                "Omega_field": 140 * u.deg**2, # Angular size of survey
        }
        return surveyspecs

nu = 2 * 115.27 * u.GHz # CO(2-1)
surveyspecs_CO21 = get_specs(nu)

nu = 1.897 * u.THz
surveyspecs_CII = get_specs(nu)

## Compute instance of SSLimPy to work with

In [ ]:
# Import Main modules. This might take some time as some functions compile before time
from SSLimPy.interface import sslimpy
from SSLimPy.cosmology import cosmology
from SSLimPy.cosmology import halo_model
from SSLimPy.cosmology import astro
from SSLimPy.LIMsurvey import covariance as scov
import SSLimPy.LIMsurvey.power_spectrum as spobs

In [ ]:
myssl = sslimpy.SSLimPy(
    settings_dict=settings,
    cosmopars=cosmodict,
    halopars=halodict,
    astropars=astrodict_CII,
    obspars_dict=surveyspecs_CII,
)

In [ ]:
myastro = myssl.current_astro
mycosmo = myastro.cosmology
h = mycosmo.h()

In [ ]:
kl = np.array([0.005303081503609659, 0.006041323271524304, 0.007187928916334343, 0.008332104669124304, 0.009409897779163538, 0.010813399895803408, 0.012534678738431336, 0.014913682931951246, 0.017138083882218533, 0.021296188723249773, 0.025118864315095794, 0.027879374059496396, 0.034643570164405786, 0.04194126347923944, 0.05211720682918039, 0.07187928916334343, 0.06364636204416534, 0.07705351461530549, 0.08332104669124309, 0.09328489365112713, 0.10175298766472778, 0.1119583794918064, 0.12534678738431337, 0.13320738474106528, 0.14033623190051728, 0.15575888520180617, 0.16267497726757185, 0.18532095446625196, 0.20214377758035074, 0.21296188723249773, 0.2282919333423334, 0.2600724444200114, 0.283680960477627, 0.33460182230795826, 0.44185830156972816, 0.5834958020234345, 0.7977867010935024, 0.965841051369878, 1.159180170826379, 1.3672535403458848]) * myastro.Mpch**-1
Pkaz = [419.54742998903936, 469.4589608686252, 514.6813856756945, 552.8452089758762, 587.8016072274912, 618.6145326496052, 631.3873231638509, 685.1707530744619, 664.4850388929378, 637.8722776129024, 599.9381905064137, 570.0555257025745, 514.6813856756945, 382.68394000528946, 338.52114318483666, 278.78299033369893, 293.3969678806872, 236.73384669888168, 207.2850818323168, 176.02004601040807, 147.9511493138171, 132.22140726355664, 121.84250425531785, 110.00694861616438, 92.46477801343816, 81.79408406061825, 68.05189710639766, 59.58650719889085, 52.71005763942232, 47.106075419832756, 40.82693201054089, 35.02504470058053, 27.689026841957112, 19.56230916853349, 11.151602437024646, 5.858028487122595, 3.1730673472016253, 1.9429496147158305, 1.2910609503014672, 0.8405357950256915]

In [ ]:
plt.loglog(kl, Pkaz)

In [ ]:
pobs_testing = spobs.PowerSpectra(myastro)
print(pobs_testing.k)
Pk_tree = (mycosmo.matpow(pobs_testing.k, 1.0) * myastro.Tbavg(1, 1, 1)**2)
print(Pk_tree)

In [ ]:
(3*myastro.Mpch**-1).to(myastro.Mpch**-1)

In [ ]:
myastro.Tbavg(1, 1, 1).to(u.uK)

In [ ]:
myastro.halomodel.halomassfunction(myastro.M, 1.0)

In [ ]:

plt.loglog(kl, Pkaz)
pobs_testing.Pk_0bs
plt.loglog(pobs_testing.k.to(myastro.Mpch**-1), Pk_tree.to(myastro.Mpch**3 * u.uK**2)/0.667**1)
plt.xlim(0.005, 50)
plt.ylim(0.7, 3000)

In [ ]:
pobs_testing.z

In [ ]:
first = np.load("/home/sefa/Desktop/LIM-Code/clusterdata/run_182753_CO21_Powerspectrum.npz")
k, Pk = first["k"], first["Pk"]
k = k * myastro.Mpch**-1

In [ ]:
I11 = myastro.Thalo(1, k, p=1, scale=(1,), beta="b1")
I20 = myastro.Thalo(1, k, p=1, scale=(2,), beta="b0")
plin = mycosmo.matpow(k, 1, tracer="clustering")
myPk = I11**2 * plin + I20

In [ ]:
myPk.unit

In [ ]:
second = np.load("/home/sefa/Desktop/LIM-Code/clusterdata/run_182453_CO21_Response_NSUB8.npz")
ks, Pks, db = second["k"], second["Pkmean"], second["deltab"]
ks = ks * myastro.Mpch**-1

In [ ]:
plt.loglog(k, Pk, label="Sims BigBox", c=Cs[0])
plt.loglog(k, myPk.to(myastro.Mpch**3 * u.uK**2), label="SSLimPy Simple", c=Cs[1])

sPk = np.median(Pks, axis=0)
smPk = np.percentile(Pks, 26, axis=0)
spPk = np.percentile(Pks, 84, axis=0)
plt.loglog(ks[0, :], sPk,label="Sims Smallbox", c=Cs[2])
plt.fill_between(ks[0, :].value, smPk, spPk, color=Cs[2], alpha=0.2)

plt.legend()
plt.xlabel(r"$k\,[h\,\mathrm{Mpc}^{-1}]$")
plt.ylabel(r"$P(k)\,[\mu\mathrm{K}^2\,\,h^{-3}\,\mathrm{Mpc}^{3}]$")

In [ ]:
plt.loglog(ks.T, Pks.T)

In [ ]:
def linear(x, r, P):
    return P * (1 + r * x)

pobs, pcov = [], []
for i in range(Pks.shape[1]):
    pi, ci = curve_fit(linear, db, Pks[:, i])
    pobs.append(pi)
    pcov.append(ci)
pobs = np.array(pobs)
pcov = np.array(pcov)

In [ ]:
myp = spobs.PowerSpectra(myastro)
myp.Pk_0bs

In [ ]:
plt.loglog(k[1:], Pk[1:], label="Sims BigBox", c=Cs[0])
plt.loglog(k, myPk.to(myastro.Mpch**3 * u.uK**2), label="SSLimPy Simple", c=Cs[1])
plt.loglog(k, I11**2 * plin.to(myastro.Mpch**3) + 65.7 * myastro.Mpch**3 * u.uK**2, "k--", label="SSLimPy Simple")
plt.loglog(myp.k, myp.Pk_0bs.to(myastro.Mpch**3 * u.uK**2), label="SSLimPy Full", c=Cs[2])

sPk = np.median(Pks, axis=0)
smPk = np.percentile(Pks, 26, axis=0)
spPk = np.percentile(Pks, 84, axis=0)
plt.loglog(ks[0, 1:], sPk[1:],label="Sims Smallbox", c=Cs[3])
plt.fill_between(ks[0, 1:].value, smPk[1:], spPk[1:], color=Cs[3], alpha=0.2)

plt.legend()
plt.xlabel(r"$k\,[h\,\mathrm{Mpc}^{-1}]$")
plt.ylabel(r"$P(k)\,[\mu\mathrm{K}^2\,\,h^{-3}\,\mathrm{Mpc}^{3}]$")
plt.grid(which="both")

In [ ]:
myastro.bavg(1, 1.0, 1.0), myastro.bavg("b2", 1.0, 1.0)

In [ ]:
from scipy.interpolate import UnivariateSpline

In [ ]:
from SSLimPy.utils.utils import *
from time import time

In [ ]:
ssc = scov.SuperSampleCovariance(myp)
rscc = ssc.response(myp.k, 1)/myp.Pk_0bs - 2

In [ ]:
rscc.shape

In [ ]:
plt.scatter(ks[0,:], pobs[:, 0] * 0.954059, label="Sims")
plt.semilogx(myp.k.to(myastro.Mpch**-1), rscc[:, 0], label="SSLimPy")
plt.xlim(1e-2, 3)
plt.legend()
plt.xlabel(r"$k\,[h\,\mathrm{Mpc}^{-1}]$")
plt.ylabel(r"$\mathrm{dlog}P/\mathrm{d}\delta_\mathrm{b}$")

In [ ]:
sys.path.append("../../lim")
from lim import lim

In [ ]:
m = lim(
    {
        "cosmo_input_class" : mycosmo.classcosmopars,
        "model_type" : "ML",
        "model_name": "TonyLi",
        "model_par": {
            "alpha": 1.11,
            "beta": 0.6,
            "dMF": 1, #* u.Msun * u.yr**-1 * u.Lsun**-1,
            "sig_SFR":0,
            "SFR_file": "sfr_release.dat",
            "do_quench": True,
        },
        "sigma_scatter" : 0.37,
        "hmf_model": "ST", # Sheth--Tormann halo mass function
        "bias_model": "Tinker10",
        "nu":  nu,
        "dnu": 0. * u.MHz, # Spectrograph resolution
        "nuObs": nuObs, # Observed Frequency
        "do_onehalo":True,
    }
)

In [ ]:
Mlim, LofMlim = m.M, m.LofM

In [ ]:
LofMs = myastro.massluminosityfunction(Mlim, 1)

In [ ]:
kl, P1h = m.k, m.Pk_onehalo[0, :]

In [ ]:
kl.unit

In [ ]:
m.ft_NFW.shape

In [ ]:
plt.semilogx(kl, m.ft_NFW[180, :], c=Cp[0])
plt.semilogx(kl, m.ft_NFW[270, :], c=Cp[1])
plt.semilogx(kl, myastro.halomodel.ft_NFW(kl, m.M[180], 1.0), c=Cp[2])
plt.semilogx(kl, myastro.halomodel.ft_NFW(kl, m.M[270], 1.0), c=Cp[3])

In [ ]:
sn_lim = np.trapz(m.LofM**2 * m.dndM, x=m.M)
sn_ssl = np.trapz(myastro.massluminosityfunction(m.M, 1)**2 * myastro.halomodel.halomassfunction(m.M, 1.0), x=m.M) 

In [ ]:
m.CLT**2 * sn_lim, m.CLT**2 * sn_ssl

In [ ]:
sig_SFR = m.model_par['sig_SFR']
alpha = 1.11
Sfactor_lim = np.exp(m.sigma_scatter**2*np.log(10)**2) * np.exp((2.*alpha**-2-alpha**-1) *sig_SFR**2*np.log(10)**2)

In [ ]:
m.CLT**2 * sn_lim * Sfactor_lim

In [ ]:
plt.loglog(kl, P1h, label="lim")
plt.loglog(k, I20, label="SLimpy")
plt.xlabel(r"$k\,[\mathrm{Mpc}^{-1}]$")
plt.ylabel(r"$P(k)\,[\mu\mathrm{K}^2\,\mathrm{Mpc}^{3}]$")
plt.legend()

In [ ]:
myastro.Tavg(1.0)

In [ ]:
myastro.bavg(1, 1, 1)

In [ ]:
raw4 = np.load("/home/sefa/Desktop/LIM-Code/clusterdata/run_167882_CO21_Response_NSUB4.npz")
ke = raw4["kedges"]
Pk = raw4["Pkmean"]
db = raw4["deltab"]

kc = np.sqrt(ke[1:], ke[:-1])
kmask = np.any(np.isnan(Pk), axis=0)
k4, Pk4 = kc[~kmask], Pk[:, ~kmask]

In [ ]:
raw8 = np.load("/home/sefa/Desktop/LIM-Code/clusterdata/run_167879_CO21_Response_NSUB8.npz")
ke = raw8["kedges"]
Pk = raw8["Pkmean"]
db = raw8["deltab"]

kc = np.sqrt(ke[1:], ke[:-1])
kmask = np.any(np.isnan(Pk), axis=0)
k8, Pk8 = kc[~kmask], Pk[:, ~kmask]

In [ ]:
Ptest = myssl.current_cosmology.matpow(k4 * h * u.Mpc**-1, 1, nonlinear=False)

In [ ]:
Tb = myssl.current_astro.Tbavg("b1", 1, 1).to(u.uK)
T02 = myssl.current_astro.Thalo(1, k4 * h * u.Mpc**-1, p=1, scale=(2,), beta="b0")

In [ ]:
raw8 = np.load("/home/sefa/Desktop/LIM-Code/clusterdata/run_167879_CO21_Response_NSUB8.npz")
raw8 = np.load("/home/sefa/Desktop/LIM-Code/clusterdata/run_167882_CO21_Response_NSUB4.npz")
ke = raw8["kedges"]
Pk = raw8["Pkmean"]
db = raw8["deltab"]

kc = np.sqrt(ke[1:], ke[:-1])
kmask = np.any(np.isnan(Pk), axis=0)

k, P = kc[~kmask], Pk[:,~kmask]


def f(x, a, b):
    return a * x  + b

r, s = [], []
for ik in range(len(k)):
    pmean, cov= curve_fit(f, db, P[:, ik])
    r.append(pmean[0]/pmean[1])
    s.append(
        np.sqrt(cov[0,0]/pmean[1]**2+pmean[0]**2/pmean[1]**4*cov[1,1]-2*pmean[0]/pmean[1]**3*cov[0, 1])
    )